# Análisis de comunidades de primates

Este notebook **solo orquesta y explora**. Toda la lógica vive en `spatialcom`,
bajo control de versiones y con pruebas. Si algo hay que corregir, se corrige en
el módulo, no aquí.

In [ ]:
%load_ext autoreload
%autoreload 2

from spatialcom import Config, Pipeline
from spatialcom.viz.theme import apply_theme, save_figure

apply_theme()
cfg = Config.from_yaml('../configs/primates_colombia.yaml')
cfg.run_id

## Ejecución completa

Cada paso persiste sus salidas en `cfg.run_dir`; se puede reanudar desde cualquiera.

In [ ]:
pipe = Pipeline(cfg).run_all()
pipe.summary()

## Exploración de resultados

In [ ]:
catalogo = pipe.state.catalog
catalogo.sort_values('n_cells', ascending=False).head(10)

In [ ]:
clu = pipe.state.clusters
print(f'k = {clu.k}, r cofenética = {clu.cophenetic_r:.3f}')
clu.diagnostics

## Figuras

Las funciones de `viz` devuelven `(fig, ax)`: se pueden ajustar antes de guardar.

In [ ]:
from spatialcom.viz.dendrogram import plot_dendrogram, plot_k_diagnostics

fig, ax = plot_dendrogram(clu, truncate_at=40)
save_figure(fig, cfg.run_dir / 'figuras' / 'fig1_dendrograma', fmt=cfg.output.figure_format)

fig, ax = plot_k_diagnostics(clu)
save_figure(fig, cfg.run_dir / 'figuras' / 'figS1_silueta', fmt=cfg.output.figure_format)

In [ ]:
from spatialcom.viz.maps import plot_cluster_map

fig, ax = plot_cluster_map(pipe.state.grid, clu.labels)
save_figure(fig, cfg.run_dir / 'figuras' / 'fig2_mapa_clusters', fmt=cfg.output.figure_format)

## Tablas del manuscrito

In [ ]:
from spatialcom.viz.tables import cluster_summary_table, export_table

tabla = cluster_summary_table(catalogo, clu.labels, extra_means=['pct_loss_total'])
export_table(tabla, cfg.run_dir / 'tablas' / 'tabla1_clusters', fmt='tex',
             caption='Resumen de los grupos de comunidades.')
tabla